# FP-Growth linguístico sem autoria

Uma transação por notícia, com atributos extraídos da mesma janela do baseline: os primeiros **300 caracteres após normalização NFKC**. Este notebook executa a comparação controlada entre estilo legado, estilo corrigido, POS e dependências sintáticas. Os rótulos Fake/True e a autoria não participam da descoberta das variantes linguísticas.

Os módulos `linguistic_features.py` e `linguistic_fp_growth.py`, distribuídos ao lado deste notebook, concentram a extração e a mineração para que o mesmo experimento também seja executável por Python. Os notebooks anteriores são preservados. Consulte [o protocolo e glossário](../../../docs/modelos/fp-growth-linguistico.md).

O teste canônico já foi observado em análises anteriores. As métricas deste notebook são **evidência exploratória de estabilidade**, não uma nova avaliação confirmatória nem desempenho de um classificador.

## 1. Ambiente e configuração

Use kernel Python 3 com as dependências de `machine-learning/requirements-linguistic.txt`. O ambiente isolado desta execução fica em `Eldorado/.codex-analysis/metadados-venv/`. A primeira execução extrai os 7.200 documentos; as seguintes verificam e reutilizam o cache. Não é necessário executar os scripts da equipe nem baixar `master`.

`OUTPUT_DIR=None` cria uma nova pasta de resultados. Para ler um resultado já concluído sem executar a mineração novamente, indique seu caminho em `EXISTING_RUN`. Os 100 bootstraps reamostram grupos pareados, com seed 42, e quantificam as métricas das regras já congeladas.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

search = [Path.cwd(), *Path.cwd().parents]
ROOT = next(candidate for parent in search
            for candidate in (parent / 'machine-learning',
                              parent / 'olimpo-fake-news-ai' / 'machine-learning', parent)
            if (candidate / 'unsupervised-learning/history/mineracao-de-padroes/fp-growth-linguistico/linguistic_fp_growth.py').is_file())
MODULE_DIR = ROOT / 'unsupervised-learning/history/mineracao-de-padroes/fp-growth-linguistico'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from linguistic_fp_growth import run_experiment

SEED = 42
BOOTSTRAP_REPETITIONS = 100
FORCE_EXTRACT = False
OUTPUT_DIR = None
EXISTING_RUN = ROOT / 'outputs/model-comparison/fp-growth-linguistic-20261006T230752Z'  # Ex.: ROOT / 'outputs/model-comparison/fp-growth-linguistic-<UTC>'
pd.set_option('display.max_colwidth', 110)
pd.set_option('display.max_columns', 20)

In [2]:
if EXISTING_RUN is None:
    RUN = Path(run_experiment(output_dir=OUTPUT_DIR,
                              force_extract=FORCE_EXTRACT,
                              bootstrap_repetitions=BOOTSTRAP_REPETITIONS))
else:
    RUN = Path(EXISTING_RUN)
    if not (RUN / 'run_manifest.json').is_file():
        raise FileNotFoundError(f'Run inexistente ou incompleto: {RUN}')
manifest = json.loads((RUN / 'run_manifest.json').read_text(encoding='utf-8'))
display(Markdown(f'**Resultados desta execução:** `{RUN}`'))
display({key: manifest[key] for key in ('status', 'created_utc', 'corpus', 'parameters',
                                       'bootstrap', 'environment', 'source_sha256')
         if key in manifest})

**Resultados desta execução:** `C:\Users\CUL7CA\Desktop\Eldorado\olimpo-fake-news-ai\machine-learning\outputs\model-comparison\fp-growth-linguistic-20261006T230752Z`

{'status': 'executed',
 'created_utc': '2026-10-06T23:08:42.346376+00:00',
 'corpus': {'name': 'Fake.br-Corpus',
  'revision': '780f5516c4ae070761632d98ac3368f3ded09d35',
  'url': 'https://codeload.github.com/roneysco/Fake.br-Corpus/zip/780f5516c4ae070761632d98ac3368f3ded09d35',
  'archive_sha256': 'be91c188f621424017bd79a0f33528dcc27f8a6151adb2bcb899c17719eb4090',
  'expected_sha256': 'be91c188f621424017bd79a0f33528dcc27f8a6151adb2bcb899c17719eb4090',
  'record_count': 7200,
  'labels': {'0': 'True', '1': 'Fake'}},
 'parameters': {'min_support': 0.08,
  'max_len': 3,
  'min_confidence': 0.5,
  'min_lift': 1.05,
  'min_jaccard': 0.1,
  'redundancy_jaccard': 0.9,
  'consolidation_jaccard': 0.75,
  'extension_tolerance': 0.03,
  'low_quantile': 0.25,
  'high_quantile': 0.75,
  'character_limit': 300},
 'bootstrap': {'repetitions': 100,
  'seed': 42,
  'unit': 'canonical paired/exact-duplicate group',
  'interval': 'percentile 2.5%-97.5%; fixed validation rules and thresholds'},
 'environ

## 2. Comparações que separam as alterações

| Variante | O que muda |
|---|---|
| `legacy_author` | Reprodução do baseline, incluindo autoria. |
| `legacy_style` | Estilo legado sem autoria: referência do principal anterior. |
| `legacy_pos` | Estilo legado + seis taxas POS, isolando a adição gramatical. |
| `corrected_style` | Remove TTR redundante e corrige a medida de pontuação; sem POS/DEP. |
| `corrected_pos` | Estilo corrigido + seis taxas POS. |
| `corrected_pos_dep` | Acrescenta seis taxas de dependências à janela de 300 caracteres. |
| `corrected_pos_dep_complete` | Sensibilidade: DEP calculada descartando a última frase analisada potencialmente truncada. |

A identidade antiga `typeTokenRatio = diversidade × (1 − punctuationDensity)` favorece associações estruturais. A correção remove esse trio redundante, mas isso não torna os novos atributos estatisticamente independentes. ADJ e `amod`, por exemplo, descrevem aspectos relacionados dos mesmos tokens. A mudança de estilo não deve ser atribuída ao acréscimo de POS.

A variante `complete` continua limitada aos 300 caracteres. Ela não recupera o texto completo nem garante frases verdadeiramente completas: é uma análise de sensibilidade ao corte.

In [3]:
summary = pd.read_csv(RUN / 'variant_summary.csv')
comparisons = pd.read_csv(RUN / 'comparisons.csv')
display(summary)
display(comparisons)

# Regras são direcionais; padrões consolidados são conjuntos de itens.
# Não comparar suas contagens como se fossem a mesma unidade.
summary.plot.bar(x='variant', y=['eligible_rules', 'validation_pass_rules'],
                 figsize=(12, 4), rot=30)
plt.ylabel('Quantidade de regras')
plt.title('Regras elegíveis no treino e regras que mantêm os filtros na validação')
plt.tight_layout()
plt.show()

,variant,feature_count,item_count,frequent_itemsets,all_rules,eligible_rules,patterns,primary_patterns,validation_pass_rules,test_pass_rules
0,legacy_author,6,10,41,82,28,31,30,27,28
1,legacy_style,5,8,18,20,8,10,10,8,8
2,legacy_pos,11,20,85,134,18,65,65,13,14
3,corrected_style,4,6,11,10,0,5,5,0,0
4,corrected_pos,10,18,75,118,10,57,57,5,6
5,corrected_pos_dep,16,30,285,802,152,255,196,121,120
6,corrected_pos_dep_complete,16,30,303,934,172,273,240,138,135


,comparison,new_variant,old_variant,new_rules,retained_rules,removed_rules,new_patterns,retained_patterns,removed_patterns
0,legacy_style_vs_legacy_author,legacy_style,legacy_author,0,8,20,0,10,21
1,legacy_pos_vs_legacy_style,legacy_pos,legacy_style,10,8,0,55,10,0
2,corrected_style_vs_legacy_style,corrected_style,legacy_style,0,0,8,3,2,8
3,corrected_pos_vs_corrected_style,corrected_pos,corrected_style,10,0,0,52,5,0
4,corrected_pos_vs_legacy_pos,corrected_pos,legacy_pos,0,10,8,9,48,17
5,corrected_pos_dep_vs_corrected_pos,corrected_pos_dep,corrected_pos,142,10,0,198,57,0
6,corrected_pos_dep_complete_vs_corrected_pos_dep,corrected_pos_dep_complete,corrected_pos_dep,162,10,142,216,57,198


C:\Users\CUL7CA\AppData\Local\Temp\ipykernel_27868\2718109463.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Auditoria dos atributos por notícia

POS/DEP são anotações **estimadas pelo modelo**, e não etiquetas verificadas manualmente. Zero indica tag ausente em um documento válido; taxas sem denominador são ausentes. A pontuação é contada com `token.is_punct`; espaços são excluídos dos denominadores. Tokens numéricos continuam no conjunto lexical.

Cada taxa POS/DEP usa contagem lexical por documento dividida pelo número de tokens lexicais elegíveis. As contagens e denominadores são preservados sem arredondamento. As distribuições abaixo usam somente o treino; não há seleção de atributos por classe.

In [4]:
features = pd.read_csv(RUN / 'linguistic_features.csv').set_index('record_id')
splits = pd.read_csv(RUN / 'split_assignments.csv').set_index('record_id')
assert features.index.is_unique and splits.index.is_unique
assert set(features.index) == set(splits.index)
train_ids = splits.index[splits['partition'].eq('train')]
train_features = features.loc[train_ids]
assert len(train_features) == 4320
display(pd.read_csv(RUN / 'feature_audit.csv'))
audit = json.loads((RUN / 'feature_audit.json').read_text(encoding='utf-8'))
display({key: value for key, value in audit.items() if key != 'distributions'})

sample_columns = [c for c in features.columns
                  if c.startswith('POS_') and c.endswith('_rate')]
quality_columns = [c for c in features.columns
                   if any(term in c.lower() for term in
                          ('token_count', 'tokens_', 'empty', 'truncat', 'quality'))]
display(train_features[quality_columns + sample_columns].sample(n=8, random_state=SEED))
display(train_features[sample_columns].describe().T)
if sample_columns:
    train_features[sample_columns].plot.hist(bins=25, subplots=True, layout=(2, 3),
                                           figsize=(13, 7), legend=True)
    plt.suptitle('Taxas POS por notícia — treino')
    plt.tight_layout()
    plt.show()

,feature,present,missing,zeros,unique,min,max,mean,median
0,tem_autor,7200,0,3601,2,0.000000,1.000000,0.499861,0.000000
1,typeTokenRatio,7200,0,0,434,0.358974,0.921569,0.681310,0.683333
2,linkDensity,7200,0,7200,1,0.000000,0.000000,0.000000,0.000000
3,punctuationDensity,7200,0,1,364,0.000000,0.470588,0.156761,0.152542
4,uppercaseRatio,7200,0,3156,134,0.000000,0.420000,0.021394,0.020000
5,diversidade,7200,0,0,300,0.545455,1.000000,0.808123,0.812500
6,text_chars_raw,7200,0,0,4537,46.000000,46084.000000,3900.797778,2187.000000
7,text_chars_normalized,7200,0,0,4538,46.000000,46084.000000,3900.837778,2187.000000
8,text_chars_window,7200,0,0,85,46.000000,300.000000,298.164167,300.000000
9,tokens_all,7200,0,0,67,11.000000,82.000000,57.831250,58.000000


{'records': 7200,
 'unique_record_ids': True,
 'quality': {'quality_empty': 0,
  'quality_noeligible': 0,
  'quality_truncated': 7101,
  'quality_complete_last_sentence_dropped': 6988,
  'quality_complete_noeligible': 126},
 'legacy_identity_max_abs_error': 1.1102230246251565e-16,
 'denominators': {'POS_and_DEP': 'tokens_lexical: excludes space/punctuation, includes NUM',
  'punctuationDensity_spacy': 'tokens_nonspace',
  'DEP_complete': 'tokens_complete_lexical'},
 'warnings': ['Legacy typeTokenRatio = diversidade * (1 - punctuationDensity); rules between them may be mathematical.',
  'Legacy punctuationDensity includes numeric regex tokens; spaCy punctuation is a separately versioned measurement.',
  'POS rates and DEP rates are compositions over the same lexical tokens; tags within each system compete for mass.',
  'POS and DEP are model predictions; ADJ/amod and ADV/advmod may be structurally redundant.',
  'DEP_complete drops the possibly clipped final predicted sentence without r

,tokens_all,tokens_nonspace,tokens_lexical,quality_empty,quality_noeligible,quality_truncated,quality_complete_last_sentence_dropped,quality_complete_noeligible,tokens_complete_lexical,POS_ADJ_rate,POS_ADV_rate,POS_NOUN_rate,POS_PROPN_rate,POS_VERB_rate,POS_PRON_rate
record_id,,,,,,,,,,,,,,,
fake/1692,60,58,50,False,False,True,True,False,44,0.060000,0.080000,0.220000,0.120000,0.140000,0.020000
fake/1165,68,66,52,False,False,True,True,False,40,0.019231,0.076923,0.230769,0.115385,0.076923,0.057692
true/1197,56,55,45,False,False,True,True,False,43,0.000000,0.022222,0.222222,0.288889,0.044444,0.000000
true/1648,57,56,51,False,False,True,True,False,10,0.039216,0.019608,0.333333,0.098039,0.117647,0.000000
fake/1231,57,55,46,False,False,True,True,False,18,0.043478,0.021739,0.130435,0.239130,0.152174,0.065217
true/1789,56,55,51,False,False,True,True,False,46,0.019608,0.000000,0.274510,0.196078,0.137255,0.039216
true/405,59,58,49,False,False,True,True,False,10,0.061224,0.061224,0.204082,0.122449,0.142857,0.020408
true/2493,58,57,50,False,False,True,True,False,36,0.100000,0.020000,0.220000,0.140000,0.100000,0.020000


,count,mean,std,min,25%,50%,75%,max
POS_ADJ_rate,4320.0,0.049378,0.034683,0.000000,0.020833,0.042553,0.068182,0.240000
POS_ADV_rate,4320.0,0.028854,0.027833,0.000000,0.000000,0.021277,0.041667,0.200000
POS_NOUN_rate,4320.0,0.229480,0.054770,0.058824,0.191489,0.226415,0.265306,0.452381
POS_PROPN_rate,4320.0,0.163426,0.080653,0.000000,0.104167,0.159091,0.215686,0.604651
POS_VERB_rate,4320.0,0.125157,0.043052,0.000000,0.096154,0.122449,0.152174,0.333333
POS_PRON_rate,4320.0,0.026682,0.027726,0.000000,0.000000,0.020833,0.040816,0.210526


C:\Users\CUL7CA\AppData\Local\Temp\ipykernel_27868\841391171.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Discretização congelada no treino

`baixo` e `alto` significam faixas definidas pelos quantis 25% e 75% do **treino**, não quantidades absolutas ou juízos de qualidade. Empates e zeros podem fazer um item cobrir mais de 25% dos registros. A discretização registra limites, coberturas e atributos omitidos; cortes coincidentes não criam faixas artificiais. Os mesmos limites são aplicados na validação e no teste.

In [5]:
for variant in ['legacy_style', 'corrected_style', 'corrected_pos_dep',
                'corrected_pos_dep_complete']:
    display(Markdown(f'**{variant}**'))
    display(pd.read_csv(RUN / variant / 'discretization.csv'))

**legacy_style**

,feature,item,direction,threshold,low_threshold,high_threshold,low_quantile,high_quantile,missing,omitted,omission_reason,criterion,train_occurrences,train_actual_support,validation_missing,validation_actual_support,test_missing,test_actual_support
0,typeTokenRatio,typeTokenRatio_baixo,baixo,0.636364,0.636364,0.730769,0.25,0.75,0,False,NaN,value <= threshold,1097,0.253935,0,0.259028,0,0.257639
1,typeTokenRatio,typeTokenRatio_alto,alto,0.730769,0.636364,0.730769,0.25,0.75,0,False,NaN,value >= threshold,1090,0.252315,0,0.250000,0,0.247917
2,linkDensity,linkDensity_baixo,baixo,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value <= threshold,0,0.000000,0,NaN,0,NaN
3,linkDensity,linkDensity_alto,alto,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value >= threshold,0,0.000000,0,NaN,0,NaN
4,punctuationDensity,punctuationDensity_baixo,baixo,0.120000,0.120000,0.189655,0.25,0.75,0,False,NaN,value <= threshold,1094,0.253241,0,0.265278,0,0.256250
5,punctuationDensity,punctuationDensity_alto,alto,0.189655,0.120000,0.189655,0.25,0.75,0,False,NaN,value >= threshold,1102,0.255093,0,0.240972,0,0.275000
6,uppercaseRatio,uppercaseRatio_baixo,baixo,0.000000,0.000000,0.038462,0.25,0.75,0,False,NaN,value <= threshold,1882,0.435648,0,0.436806,0,0.447917
7,uppercaseRatio,uppercaseRatio_alto,alto,0.038462,0.000000,0.038462,0.25,0.75,0,False,NaN,value >= threshold,1100,0.254630,0,0.262500,0,0.232639
8,diversidade,diversidade_baixo,baixo,0.764706,0.764706,0.854167,0.25,0.75,0,False,NaN,value <= threshold,1087,0.251620,0,0.245833,0,0.240278
9,diversidade,diversidade_alto,alto,0.854167,0.764706,0.854167,0.25,0.75,0,False,NaN,value >= threshold,1110,0.256944,0,0.265278,0,0.268056


**corrected_style**

,feature,item,direction,threshold,low_threshold,high_threshold,low_quantile,high_quantile,missing,omitted,omission_reason,criterion,train_occurrences,train_actual_support,validation_missing,validation_actual_support,test_missing,test_actual_support
0,diversidade,diversidade_baixo,baixo,0.764706,0.764706,0.854167,0.25,0.75,0,False,NaN,value <= threshold,1087,0.251620,0,0.245833,0,0.240278
1,diversidade,diversidade_alto,alto,0.854167,0.764706,0.854167,0.25,0.75,0,False,NaN,value >= threshold,1110,0.256944,0,0.265278,0,0.268056
2,linkDensity,linkDensity_baixo,baixo,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value <= threshold,0,0.000000,0,NaN,0,NaN
3,linkDensity,linkDensity_alto,alto,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value >= threshold,0,0.000000,0,NaN,0,NaN
4,uppercaseRatio,uppercaseRatio_baixo,baixo,0.000000,0.000000,0.038462,0.25,0.75,0,False,NaN,value <= threshold,1882,0.435648,0,0.436806,0,0.447917
5,uppercaseRatio,uppercaseRatio_alto,alto,0.038462,0.000000,0.038462,0.25,0.75,0,False,NaN,value >= threshold,1100,0.254630,0,0.262500,0,0.232639
6,punctuationDensity_spacy,punctuationDensity_spacy_baixo,baixo,0.096154,0.096154,0.158730,0.25,0.75,0,False,NaN,value <= threshold,1085,0.251157,0,0.247917,0,0.233333
7,punctuationDensity_spacy,punctuationDensity_spacy_alto,alto,0.158730,0.096154,0.158730,0.25,0.75,0,False,NaN,value >= threshold,1091,0.252546,0,0.246528,0,0.258333


**corrected_pos_dep**

,feature,item,direction,threshold,low_threshold,high_threshold,low_quantile,high_quantile,missing,omitted,omission_reason,criterion,train_occurrences,train_actual_support,validation_missing,validation_actual_support,test_missing,test_actual_support
0,diversidade,diversidade_baixo,baixo,0.764706,0.764706,0.854167,0.25,0.75,0,False,NaN,value <= threshold,1087,0.251620,0,0.245833,0,0.240278
1,diversidade,diversidade_alto,alto,0.854167,0.764706,0.854167,0.25,0.75,0,False,NaN,value >= threshold,1110,0.256944,0,0.265278,0,0.268056
2,linkDensity,linkDensity_baixo,baixo,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value <= threshold,0,0.000000,0,NaN,0,NaN
3,linkDensity,linkDensity_alto,alto,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value >= threshold,0,0.000000,0,NaN,0,NaN
4,uppercaseRatio,uppercaseRatio_baixo,baixo,0.000000,0.000000,0.038462,0.25,0.75,0,False,NaN,value <= threshold,1882,0.435648,0,0.436806,0,0.447917
5,uppercaseRatio,uppercaseRatio_alto,alto,0.038462,0.000000,0.038462,0.25,0.75,0,False,NaN,value >= threshold,1100,0.254630,0,0.262500,0,0.232639
6,punctuationDensity_spacy,punctuationDensity_spacy_baixo,baixo,0.096154,0.096154,0.158730,0.25,0.75,0,False,NaN,value <= threshold,1085,0.251157,0,0.247917,0,0.233333
7,punctuationDensity_spacy,punctuationDensity_spacy_alto,alto,0.158730,0.096154,0.158730,0.25,0.75,0,False,NaN,value >= threshold,1091,0.252546,0,0.246528,0,0.258333
8,POS_ADJ_rate,POS_ADJ_rate_baixo,baixo,0.020833,0.020833,0.068182,0.25,0.75,0,False,NaN,value <= threshold,1128,0.261111,0,0.283333,0,0.263194
9,POS_ADJ_rate,POS_ADJ_rate_alto,alto,0.068182,0.020833,0.068182,0.25,0.75,0,False,NaN,value >= threshold,1089,0.252083,0,0.243056,0,0.251389


**corrected_pos_dep_complete**

,feature,item,direction,threshold,low_threshold,high_threshold,low_quantile,high_quantile,missing,omitted,omission_reason,criterion,train_occurrences,train_actual_support,validation_missing,validation_actual_support,test_missing,test_actual_support
0,diversidade,diversidade_baixo,baixo,0.764706,0.764706,0.854167,0.25,0.75,0,False,NaN,value <= threshold,1087,0.251620,0,0.245833,0,0.240278
1,diversidade,diversidade_alto,alto,0.854167,0.764706,0.854167,0.25,0.75,0,False,NaN,value >= threshold,1110,0.256944,0,0.265278,0,0.268056
2,linkDensity,linkDensity_baixo,baixo,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value <= threshold,0,0.000000,0,NaN,0,NaN
3,linkDensity,linkDensity_alto,alto,0.000000,0.000000,0.000000,0.25,0.75,0,True,equal_quantiles,value >= threshold,0,0.000000,0,NaN,0,NaN
4,uppercaseRatio,uppercaseRatio_baixo,baixo,0.000000,0.000000,0.038462,0.25,0.75,0,False,NaN,value <= threshold,1882,0.435648,0,0.436806,0,0.447917
5,uppercaseRatio,uppercaseRatio_alto,alto,0.038462,0.000000,0.038462,0.25,0.75,0,False,NaN,value >= threshold,1100,0.254630,0,0.262500,0,0.232639
6,punctuationDensity_spacy,punctuationDensity_spacy_baixo,baixo,0.096154,0.096154,0.158730,0.25,0.75,0,False,NaN,value <= threshold,1085,0.251157,0,0.247917,0,0.233333
7,punctuationDensity_spacy,punctuationDensity_spacy_alto,alto,0.158730,0.096154,0.158730,0.25,0.75,0,False,NaN,value >= threshold,1091,0.252546,0,0.246528,0,0.258333
8,POS_ADJ_rate,POS_ADJ_rate_baixo,baixo,0.020833,0.020833,0.068182,0.25,0.75,0,False,NaN,value <= threshold,1128,0.261111,0,0.283333,0,0.263194
9,POS_ADJ_rate,POS_ADJ_rate_alto,alto,0.068182,0.020833,0.068182,0.25,0.75,0,False,NaN,value >= threshold,1089,0.252083,0,0.243056,0,0.251389


## 5. Regras congeladas e verificação dos filtros

Suporte é a fração de notícias que contém antecedente e consequente; confidence é a fração com consequente entre as notícias com antecedente; lift divide confidence pela frequência do consequente. `A → B` não indica causalidade nem uma decisão Fake/True.

Mantemos suporte ≥0,08, confidence ≥0,50, lift ≥1,05 e Jaccard ≥0,10. A auditoria de similaridade entre coberturas dos padrões do baseline também é preservada; ela é distinta do Jaccard da regra. As regras são descobertas somente no treino; contar quais mantêm os filtros na validação é avaliação posterior, sem alterar os quantis ou minerar novas regras nessa partição.

Os intervalos de 95% abaixo vêm do bootstrap dos pares da validação. Eles medem variação das métricas das regras congeladas, sem redescobri-las; não corrigem a seleção entre muitas regras nem substituem uma avaliação externa.

In [6]:
metrics = {}
for variant in summary['variant']:
    frame = pd.read_csv(RUN / variant / 'rule_metrics.csv')
    metrics[variant] = frame
    if not frame.empty:
        for split in ('train', 'validation', 'test'):
            # Verificação independente dos quatro filtros mínimos.
            minimum = ((frame[f'{split}_support'] >= .08) &
                       (frame[f'{split}_confidence'] >= .50) &
                       (frame[f'{split}_lift'] >= 1.05) &
                       (frame[f'{split}_jaccard'] >= .10))
            saved = frame[f'{split}_passes_filters'].astype(str).str.lower().eq('true')
            assert saved.equals(minimum), (variant, split)
        assert frame['train_passes_filters'].astype(str).str.lower().eq('true').all()
display(Markdown('Os filtros mínimos registrados foram conferidos nas três partições.'))

Os filtros mínimos registrados foram conferidos nas três partições.

In [7]:
def rule_text(frame):
    return frame['antecedents'].map(lambda value: ' + '.join(json.loads(value))) + ' → ' +            frame['consequents'].map(lambda value: ' + '.join(json.loads(value)))

def readable_rules(frame, limit=None):
    if frame.empty:
        return frame
    shown = frame.copy()
    shown.insert(0, 'regra', rule_text(shown))
    columns = ['regra', 'train_occurrences', 'train_support', 'train_confidence',
               'train_lift', 'validation_occurrences', 'validation_support',
               'validation_confidence', 'validation_lift',
               'validation_confidence_ci_low', 'validation_confidence_ci_high',
               'validation_lift_ci_low', 'validation_lift_ci_high',
               'validation_confidence_bootstrap_valid_repetitions',
               'validation_lift_bootstrap_valid_repetitions',
               'validation_bootstrap_pass_fraction', 'test_support',
               'test_confidence', 'test_lift']
    shown = shown[[column for column in columns if column in shown]]
    return shown if limit is None else shown.head(limit)

display(Markdown('**Regras anteriores sem autoria — referência**'))
display(readable_rules(metrics['legacy_style']))
display(Markdown('**Estilo corrigido sem POS/DEP — efeito da correção lexical**'))
display(readable_rules(metrics['corrected_style']))

**Regras anteriores sem autoria — referência**

,regra,train_occurrences,train_support,train_confidence,train_lift,validation_occurrences,validation_support,validation_confidence,validation_lift,validation_confidence_ci_low,validation_confidence_ci_high,validation_lift_ci_low,validation_lift_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction,test_support,test_confidence,test_lift
0,diversidade_baixo → typeTokenRatio_baixo,743,0.171991,0.683533,2.691760,251,0.174306,0.709040,2.737311,0.660061,0.749026,2.514108,2.915769,100,100,1.00,0.172222,0.716763,2.782045
1,typeTokenRatio_baixo → diversidade_baixo,743,0.171991,0.677302,2.691760,251,0.174306,0.672922,2.737311,0.618626,0.706713,2.514108,2.915769,100,100,1.00,0.172222,0.668464,2.782045
2,typeTokenRatio_alto → diversidade_alto,716,0.165741,0.656881,2.556509,249,0.172917,0.691667,2.607330,0.628228,0.732418,2.412691,2.834887,100,100,1.00,0.162500,0.655462,2.445248
3,diversidade_alto → typeTokenRatio_alto,716,0.165741,0.645045,2.556509,249,0.172917,0.651832,2.607330,0.605403,0.691931,2.412691,2.834887,100,100,1.00,0.162500,0.606218,2.445248
4,typeTokenRatio_alto → punctuationDensity_baixo,597,0.138194,0.547706,2.162790,193,0.134028,0.536111,2.020942,0.492166,0.602528,1.885261,2.207245,100,100,0.95,0.130556,0.526611,2.055066
5,punctuationDensity_baixo → typeTokenRatio_alto,597,0.138194,0.545704,2.162790,193,0.134028,0.505236,2.020942,0.459025,0.564096,1.885261,2.207245,100,100,0.50,0.130556,0.509485,2.055066
6,typeTokenRatio_baixo → punctuationDensity_alto,598,0.138426,0.545123,2.136962,199,0.138194,0.533512,2.213998,0.494045,0.587191,2.024239,2.461937,100,100,0.94,0.145833,0.566038,2.058319
7,punctuationDensity_alto → typeTokenRatio_baixo,598,0.138426,0.542650,2.136962,199,0.138194,0.573487,2.213998,0.516105,0.636351,2.024239,2.461937,100,100,1.00,0.145833,0.530303,2.058319


**Estilo corrigido sem POS/DEP — efeito da correção lexical**

,variant,rule_id,pattern_id,antecedents,consequents,features,train_occurrences,train_antecedent_occurrences,train_consequent_occurrences,train_support,...,validation_lift,validation_jaccard,validation_passes_filters,validation_confidence_ci_low,validation_confidence_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_ci_low,validation_lift_ci_high,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction


## 6. Regras linguísticas novas e diferença para as anteriores

A tabela mostra regras elegíveis no treino com POS/DEP e que mantêm os filtros na validação. A ordem é suporte e confidence na validação, sem usar os rótulos de classe. Regras inversas compartilham a ocorrência conjunta, mas sua confidence pode mudar. O arquivo `rule_metrics.csv` contém **todas** as regras elegíveis, inclusive as que não mantiveram os filtros fora do treino.

As comparações entre representações usam igualdade exata de itens e direção. Um item renomeado ou redefinido não é tratado como equivalente ao antigo; ausência de igualdade não prova que todo o perfil linguístico desapareceu.

In [8]:
for variant in ['legacy_pos', 'corrected_pos', 'corrected_pos_dep',
                'corrected_pos_dep_complete']:
    frame = metrics[variant]
    display(Markdown(f'**{variant}**'))
    if frame.empty:
        display(Markdown('Nenhuma regra elegível no treino.'))
        continue
    linguistic = (frame['antecedents'] + frame['consequents']).str.contains('POS_|DEP_', regex=True)
    validation_pass = frame['validation_passes_filters'].astype(str).str.lower().eq('true')
    selected = frame[linguistic & validation_pass].sort_values(
        ['validation_support', 'validation_confidence'], ascending=False)
    display(readable_rules(selected, limit=12))
    display(Markdown(f'{len(selected)} regras linguísticas mantêm os filtros na validação; '
                     f'{int(linguistic.sum())} foram elegíveis no treino.'))

display(comparisons)
changes = pd.read_csv(RUN / 'rule_comparison.csv')
for comparison in ('corrected_style_vs_legacy_style',
                   'corrected_pos_vs_corrected_style',
                   'corrected_pos_dep_vs_corrected_pos'):
    display(Markdown(f'**Diferenças direcionais: {comparison}**'))
    changed = changes[changes['comparison'].eq(comparison)].copy()
    if changed.empty:
        display(Markdown('Não houve regras elegíveis nesta comparação.'))
        continue
    changed['regra'] = rule_text(changed)
    display(changed[['status', 'regra']].sort_values(['status', 'regra']))

**legacy_pos**

,regra,train_occurrences,train_support,train_confidence,train_lift,validation_occurrences,validation_support,validation_confidence,validation_lift,validation_confidence_ci_low,validation_confidence_ci_high,validation_lift_ci_low,validation_lift_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction,test_support,test_confidence,test_lift
12,POS_PROPN_rate_baixo → uppercaseRatio_baixo,713,0.165046,0.655331,1.504266,237,0.164583,0.688953,1.577254,0.643056,0.732421,1.478408,1.681938,100,100,1.00,0.157639,0.632312,1.411673
17,POS_ADV_rate_alto → uppercaseRatio_baixo,593,0.137269,0.502117,1.152574,204,0.141667,0.548387,1.255449,0.502765,0.611964,1.157541,1.348202,100,100,0.99,0.142361,0.535248,1.194972
14,POS_PRON_rate_alto → uppercaseRatio_baixo,631,0.146065,0.542562,1.245414,202,0.140278,0.543011,1.243141,0.499653,0.585249,1.152187,1.327578,100,100,0.97,0.145139,0.514778,1.149273
16,POS_ADJ_rate_alto → uppercaseRatio_baixo,568,0.131481,0.521579,1.197249,187,0.129861,0.534286,1.223166,0.485727,0.577933,1.129168,1.332994,100,100,0.89,0.125694,0.500000,1.116279
15,POS_NOUN_rate_alto → uppercaseRatio_baixo,576,0.133333,0.527473,1.210776,182,0.126389,0.509804,1.167119,0.471000,0.559971,1.081818,1.262678,100,100,0.73,0.123611,0.534535,1.193379


5 regras linguísticas mantêm os filtros na validação; 10 foram elegíveis no treino.

**corrected_pos**

,regra,train_occurrences,train_support,train_confidence,train_lift,validation_occurrences,validation_support,validation_confidence,validation_lift,validation_confidence_ci_low,validation_confidence_ci_high,validation_lift_ci_low,validation_lift_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction,test_support,test_confidence,test_lift
4,POS_PROPN_rate_baixo → uppercaseRatio_baixo,713,0.165046,0.655331,1.504266,237,0.164583,0.688953,1.577254,0.643056,0.732421,1.478408,1.681938,100,100,1.00,0.157639,0.632312,1.411673
9,POS_ADV_rate_alto → uppercaseRatio_baixo,593,0.137269,0.502117,1.152574,204,0.141667,0.548387,1.255449,0.502765,0.611964,1.157541,1.348202,100,100,0.99,0.142361,0.535248,1.194972
6,POS_PRON_rate_alto → uppercaseRatio_baixo,631,0.146065,0.542562,1.245414,202,0.140278,0.543011,1.243141,0.499653,0.585249,1.152187,1.327578,100,100,0.97,0.145139,0.514778,1.149273
8,POS_ADJ_rate_alto → uppercaseRatio_baixo,568,0.131481,0.521579,1.197249,187,0.129861,0.534286,1.223166,0.485727,0.577933,1.129168,1.332994,100,100,0.89,0.125694,0.500000,1.116279
7,POS_NOUN_rate_alto → uppercaseRatio_baixo,576,0.133333,0.527473,1.210776,182,0.126389,0.509804,1.167119,0.471000,0.559971,1.081818,1.262678,100,100,0.73,0.123611,0.534535,1.193379


...

,status,regra
53,removed,diversidade_alto → typeTokenRatio_alto
52,removed,diversidade_baixo → typeTokenRatio_baixo
51,removed,punctuationDensity_alto → typeTokenRatio_baixo
48,removed,punctuationDensity_baixo → typeTokenRatio_alto
49,removed,typeTokenRatio_alto → diversidade_alto
46,removed,typeTokenRatio_alto → punctuationDensity_baixo
47,removed,typeTokenRatio_baixo → diversidade_baixo
50,removed,typeTokenRatio_baixo → punctuationDensity_alto


**Diferenças direcionais: corrected_pos_vs_corrected_style**

,status,regra
56,new,POS_ADJ_rate_alto → uppercaseRatio_baixo
62,new,POS_ADV_rate_alto → uppercaseRatio_baixo
61,new,POS_NOUN_rate_alto + POS_PROPN_rate_baixo → uppercaseRatio_baixo
60,new,POS_NOUN_rate_alto + uppercaseRatio_baixo → POS_PROPN_rate_baixo
63,new,POS_NOUN_rate_alto → POS_PROPN_rate_baixo
59,new,POS_NOUN_rate_alto → uppercaseRatio_baixo
55,new,POS_PRON_rate_alto → uppercaseRatio_baixo
57,new,POS_PROPN_rate_baixo + uppercaseRatio_baixo → POS_NOUN_rate_alto
54,new,POS_PROPN_rate_baixo → POS_NOUN_rate_alto
58,new,POS_PROPN_rate_baixo → uppercaseRatio_baixo


**Diferenças direcionais: corrected_pos_dep_vs_corrected_pos**

,status,regra
90,new,DEP_advcl_rate_baixo + DEP_advmod_rate_alto → POS_ADV_rate_alto
225,new,DEP_advcl_rate_baixo + DEP_advmod_rate_baixo → DEP_ccomp_rate_baixo
128,new,DEP_advcl_rate_baixo + DEP_advmod_rate_baixo → POS_ADV_rate_baixo
152,new,DEP_advcl_rate_baixo + DEP_amod_rate_alto → DEP_ccomp_rate_baixo
160,new,DEP_advcl_rate_baixo + DEP_amod_rate_alto → POS_ADJ_rate_alto
...,...,...
145,retained,POS_NOUN_rate_alto → uppercaseRatio_baixo
109,retained,POS_PRON_rate_alto → uppercaseRatio_baixo
124,retained,POS_PROPN_rate_baixo + uppercaseRatio_baixo → POS_NOUN_rate_alto
92,retained,POS_PROPN_rate_baixo → POS_NOUN_rate_alto


## 7. Estabilidade da redescoberta no treino

Esta análise é diferente do bootstrap da validação. Reamostramos grupos inteiros do treino, respeitando pares e duplicatas, **reaprendemos os quantis e mineramos novamente** em 100 repetições com seed 42. Cada regra do run original recebe uma frequência de redescoberta por igualdade dos nomes dos itens e da direção.

Como os quantis são reestimados em cada repetição, uma regra com os mesmos nomes não garante fronteiras numéricas ou notícias cobertas idênticas. A taxa de redescoberta descreve a persistência da relação discretizada sob perturbação do treino; não é probabilidade de verdade nem resultado preditivo.

In [9]:
from linguistic_stability import write_rediscovery_stability

if not all((RUN / name).is_file() for name in
           ('training_rediscovery_stability.csv', 'training_bootstrap_summary.csv',
            'training_bootstrap_manifest.json')):
    write_rediscovery_stability(RUN, repetitions=BOOTSTRAP_REPETITIONS, seed=SEED)
training_manifest = json.loads((RUN / 'training_bootstrap_manifest.json').read_text(encoding='utf-8'))
assert training_manifest['status'] == 'executed'
assert training_manifest['repetitions'] == BOOTSTRAP_REPETITIONS
assert training_manifest['seed'] == SEED
assert training_manifest['labels_used'] is False
assert training_manifest['quantiles_refitted_each_replicate'] is True
stability = pd.read_csv(RUN / 'training_rediscovery_stability.csv')
rediscovery_runs = pd.read_csv(RUN / 'training_bootstrap_summary.csv')
assert stability['repetitions'].eq(BOOTSTRAP_REPETITIONS).all()
assert rediscovery_runs.groupby('variant')['replicate'].nunique().eq(BOOTSTRAP_REPETITIONS).all()
display(rediscovery_runs.groupby('variant')[['eligible_rules', 'retained_rules',
                                            'new_rules', 'removed_rules']].agg(['mean', 'median']))
display(stability.groupby('variant')['rediscovery_fraction'].agg(['count', 'mean', 'median', 'min', 'max']))
for variant in ['legacy_style', 'corrected_pos', 'corrected_pos_dep',
                'corrected_pos_dep_complete']:
    frame = metrics[variant].merge(stability[['variant', 'rule_id', 'rediscovery_fraction']],
                                  on=['variant', 'rule_id'], how='left')
    if frame.empty:
        continue
    linguistic = (frame['antecedents'] + frame['consequents']).str.contains('POS_|DEP_', regex=True)
    if variant != 'legacy_style':
        frame = frame[linguistic]
    frame = frame[frame['validation_passes_filters'].astype(str).str.lower().eq('true')]
    frame = frame.sort_values(['validation_support', 'validation_confidence'], ascending=False).head(12)
    shown = readable_rules(frame).copy()
    shown['rediscovery_fraction_train'] = frame['rediscovery_fraction'].to_numpy()
    display(Markdown(f'**Redescoberta no treino e métricas de validação: {variant}**'))
    display(shown)

eligible_rules        retained_rules         \
                                     mean median           mean median   
variant                                                                  
corrected_pos                        9.16    9.0           8.85    9.0   
corrected_pos_dep                  150.79  151.0         133.74  135.0   
corrected_pos_dep_complete         173.46  173.0         151.30  151.5   
corrected_style                      0.02    0.0           0.00    0.0   
legacy_author                       28.72   28.0          27.53   28.0   
legacy_pos                          18.05   18.0          16.85   17.0   
legacy_style                         8.91    8.0           8.00    8.0   

                           new_rules        removed_rules         
                                mean median          mean median  
variant                                                           
corrected_pos                   0.31    0.0          1.15    1.0  
corrected_pos_dep              17.05   17.0         18.26   17.0  
corrected_pos_dep_complete     22.16   21.0         20.70   20.5  
corrected_style                 0.02    0.0          0.00    0.0  
legacy_author                   1.19    0.0          0.47    0.0  
legacy_pos                      1.20    1.0          1.15    1.0  
legacy_style                    0.91    0.0          0.00    0.0

,count,mean,median,min,max
variant,,,,,
corrected_pos,10,0.885000,0.92,0.65,1.0
corrected_pos_dep,152,0.879868,0.98,0.33,1.0
corrected_pos_dep_complete,172,0.879651,0.98,0.38,1.0
legacy_author,28,0.983214,1.00,0.67,1.0
legacy_pos,18,0.936111,1.00,0.65,1.0
legacy_style,8,1.000000,1.00,1.00,1.0


**Redescoberta no treino e métricas de validação: legacy_style**

,regra,train_occurrences,train_support,train_confidence,train_lift,validation_occurrences,validation_support,validation_confidence,validation_lift,validation_confidence_ci_low,validation_confidence_ci_high,validation_lift_ci_low,validation_lift_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction,test_support,test_confidence,test_lift,rediscovery_fraction_train
0,diversidade_baixo → typeTokenRatio_baixo,743,0.171991,0.683533,2.691760,251,0.174306,0.709040,2.737311,0.660061,0.749026,2.514108,2.915769,100,100,1.00,0.172222,0.716763,2.782045,1.0
1,typeTokenRatio_baixo → diversidade_baixo,743,0.171991,0.677302,2.691760,251,0.174306,0.672922,2.737311,0.618626,0.706713,2.514108,2.915769,100,100,1.00,0.172222,0.668464,2.782045,1.0
2,typeTokenRatio_alto → diversidade_alto,716,0.165741,0.656881,2.556509,249,0.172917,0.691667,2.607330,0.628228,0.732418,2.412691,2.834887,100,100,1.00,0.162500,0.655462,2.445248,1.0
3,diversidade_alto → typeTokenRatio_alto,716,0.165741,0.645045,2.556509,249,0.172917,0.651832,2.607330,0.605403,0.691931,2.412691,2.834887,100,100,1.00,0.162500,0.606218,2.445248,1.0
7,punctuationDensity_alto → typeTokenRatio_baixo,598,0.138426,0.542650,2.136962,199,0.138194,0.573487,2.213998,0.516105,0.636351,2.024239,2.461937,100,100,1.00,0.145833,0.530303,2.058319,1.0
6,typeTokenRatio_baixo → punctuationDensity_alto,598,0.138426,0.545123,2.136962,199,0.138194,0.533512,2.213998,0.494045,0.587191,2.024239,2.461937,100,100,0.94,0.145833,0.566038,2.058319,1.0
4,typeTokenRatio_alto → punctuationDensity_baixo,597,0.138194,0.547706,2.162790,193,0.134028,0.536111,2.020942,0.492166,0.602528,1.885261,2.207245,100,100,0.95,0.130556,0.526611,2.055066,1.0
5,punctuationDensity_baixo → typeTokenRatio_alto,597,0.138194,0.545704,2.162790,193,0.134028,0.505236,2.020942,0.459025,0.564096,1.885261,2.207245,100,100,0.50,0.130556,0.509485,2.055066,1.0


**Redescoberta no treino e métricas de validação: corrected_pos**

,regra,train_occurrences,train_support,train_confidence,train_lift,validation_occurrences,validation_support,validation_confidence,validation_lift,validation_confidence_ci_low,validation_confidence_ci_high,validation_lift_ci_low,validation_lift_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction,test_support,test_confidence,test_lift,rediscovery_fraction_train
4,POS_PROPN_rate_baixo → uppercaseRatio_baixo,713,0.165046,0.655331,1.504266,237,0.164583,0.688953,1.577254,0.643056,0.732421,1.478408,1.681938,100,100,1.00,0.157639,0.632312,1.411673,1.00
9,POS_ADV_rate_alto → uppercaseRatio_baixo,593,0.137269,0.502117,1.152574,204,0.141667,0.548387,1.255449,0.502765,0.611964,1.157541,1.348202,100,100,0.99,0.142361,0.535248,1.194972,0.65
6,POS_PRON_rate_alto → uppercaseRatio_baixo,631,0.146065,0.542562,1.245414,202,0.140278,0.543011,1.243141,0.499653,0.585249,1.152187,1.327578,100,100,0.97,0.145139,0.514778,1.149273,1.00
8,POS_ADJ_rate_alto → uppercaseRatio_baixo,568,0.131481,0.521579,1.197249,187,0.129861,0.534286,1.223166,0.485727,0.577933,1.129168,1.332994,100,100,0.89,0.125694,0.500000,1.116279,0.95
7,POS_NOUN_rate_alto → uppercaseRatio_baixo,576,0.133333,0.527473,1.210776,182,0.126389,0.509804,1.167119,0.471000,0.559971,1.081818,1.262678,100,100,0.73,0.123611,0.534535,1.193379,0.97


**Redescoberta no treino e métricas de validação: corrected_pos_dep**

,regra,train_occurrences,train_support,train_confidence,train_lift,validation_occurrences,validation_support,validation_confidence,validation_lift,validation_confidence_ci_low,validation_confidence_ci_high,validation_lift_ci_low,validation_lift_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction,test_support,test_confidence,test_lift,rediscovery_fraction_train
33,POS_ADV_rate_baixo → DEP_advmod_rate_baixo,1231,0.284954,0.963224,2.953248,416,0.288889,0.958525,2.881579,0.937991,0.979508,2.686358,3.105252,100,100,1.00,0.295833,0.961625,2.843409,1.00
34,DEP_advmod_rate_baixo → POS_ADV_rate_baixo,1231,0.284954,0.873669,2.953248,416,0.288889,0.868476,2.881579,0.842995,0.897348,2.686358,3.105252,100,100,1.00,0.295833,0.874743,2.843409,1.00
9,DEP_advmod_rate_alto → POS_ADV_rate_alto,1008,0.233333,0.864494,3.162247,331,0.229861,0.904372,3.500793,0.874529,0.928905,3.240015,3.777614,100,100,1.00,0.226389,0.851175,3.200240,1.00
10,POS_ADV_rate_alto → DEP_advmod_rate_alto,1008,0.233333,0.853514,3.162247,331,0.229861,0.889785,3.500793,0.868424,0.913020,3.240015,3.777614,100,100,1.00,0.226389,0.851175,3.200240,1.00
57,DEP_amod_rate_baixo → POS_ADJ_rate_baixo,797,0.184491,0.728519,2.790074,299,0.207639,0.741935,2.618596,0.709139,0.779295,2.464059,2.826263,100,100,1.00,0.182639,0.751429,2.855032,1.00
58,POS_ADJ_rate_baixo → DEP_amod_rate_baixo,797,0.184491,0.706560,2.790074,299,0.207639,0.732843,2.618596,0.682867,0.775076,2.464059,2.826263,100,100,1.00,0.182639,0.693931,2.855032,1.00
140,DEP_advmod_rate_baixo → DEP_advcl_rate_baixo,896,0.207407,0.635912,1.084112,295,0.204861,0.615866,1.064643,0.572398,0.658314,1.013600,1.140482,100,100,0.67,0.218056,0.644764,1.083384,0.95
90,DEP_nsubj_rate_baixo → DEP_ccomp_rate_baixo,869,0.201157,0.784296,1.322983,294,0.204167,0.765625,1.298587,0.716399,0.802597,1.220801,1.357725,100,100,1.00,0.181250,0.763158,1.305163,1.00
88,POS_VERB_rate_baixo → DEP_ccomp_rate_baixo,904,0.209259,0.798587,1.347089,281,0.195139,0.802857,1.361736,0.762706,0.838688,1.290411,1.431367,100,100,1.00,0.195139,0.798295,1.365256,1.00
54,POS_ADJ_rate_alto → DEP_amod_rate_alto,832,0.192593,0.764004,2.865014,272,0.188889,0.777143,2.968397,0.731485,0.821688,2.752230,3.199381,100,100,1.00,0.191667,0.762431,2.935563,1.00


**Redescoberta no treino e métricas de validação: corrected_pos_dep_complete**

,regra,train_occurrences,train_support,train_confidence,train_lift,validation_occurrences,validation_support,validation_confidence,validation_lift,validation_confidence_ci_low,validation_confidence_ci_high,validation_lift_ci_low,validation_lift_ci_high,validation_confidence_bootstrap_valid_repetitions,validation_lift_bootstrap_valid_repetitions,validation_bootstrap_pass_fraction,test_support,test_confidence,test_lift,rediscovery_fraction_train
39,POS_ADV_rate_baixo → DEP_complete_advmod_rate_baixo,1202,0.278241,0.940532,2.329758,408,0.283333,0.940092,2.366666,0.918518,0.964353,2.240189,2.514631,100,100,1.00,0.286111,0.930023,2.345416,1.00
40,DEP_complete_advmod_rate_baixo → POS_ADV_rate_baixo,1202,0.278241,0.689220,2.329758,408,0.283333,0.713287,2.366666,0.690064,0.747086,2.240189,2.514631,100,100,1.00,0.286111,0.721541,2.345416,1.00
144,DEP_complete_advmod_rate_baixo → DEP_complete_ccomp_rate_baixo,1211,0.280324,0.694381,1.087251,393,0.272917,0.687063,1.075403,0.661099,0.728945,1.026122,1.121936,100,100,0.91,0.268750,0.677758,1.061993,0.99
146,DEP_complete_advmod_rate_baixo → DEP_complete_advcl_rate_baixo,1220,0.282407,0.699541,1.087057,386,0.268056,0.674825,1.065513,0.640957,0.716447,1.021639,1.119996,100,100,0.77,0.280556,0.707531,1.085031,0.99
78,DEP_complete_nsubj_rate_baixo → DEP_complete_ccomp_rate_baixo,909,0.210417,0.816712,1.278795,323,0.224306,0.813602,1.273464,0.772238,0.848703,1.207230,1.318628,100,100,1.00,0.192361,0.807580,1.265414,1.00
85,POS_VERB_rate_baixo → DEP_complete_ccomp_rate_baixo,900,0.208333,0.795053,1.244882,281,0.195139,0.802857,1.256646,0.770980,0.840662,1.209662,1.316604,100,100,1.00,0.195139,0.798295,1.250866,1.00
6,DEP_complete_advmod_rate_alto → POS_ADV_rate_alto,827,0.191435,0.777256,2.843137,278,0.193056,0.780899,3.022834,0.744300,0.820113,2.796357,3.303905,100,100,1.00,0.187500,0.758427,2.851527,1.00
7,POS_ADV_rate_alto → DEP_complete_advmod_rate_alto,827,0.191435,0.700254,2.843137,278,0.193056,0.747312,3.022834,0.708429,0.787518,2.796357,3.303905,100,100,1.00,0.187500,0.704961,2.851527,1.00
28,DEP_complete_ccomp_rate_baixo + POS_ADV_rate_baixo → DEP_complete_advmod_rate_baixo,794,0.183796,0.968293,2.398523,275,0.190972,0.968310,2.437703,0.946796,0.988225,2.294418,2.593089,100,100,1.00,0.188194,0.957597,2.414956,1.00
46,DEP_complete_advmod_rate_baixo + DEP_complete_ccomp_rate_baixo → POS_ADV_rate_baixo,794,0.183796,0.655656,2.216304,275,0.190972,0.699746,2.321736,0.656065,0.739564,2.135722,2.474970,100,100,1.00,0.188194,0.700258,2.276235,1.00


## 8. Prévia reproduzível das anotações

A amostra abaixo foi selecionada entre IDs de treino com seed 42, sem usar classe ou autoria como critério. Tokens, POS, DEP e cabeças sintáticas são **previsões do spaCy**. Esta prévia permite a revisão dos exemplos, mas sua geração não constitui validação manual com referência humana.

In [10]:
if not (RUN / 'annotation_preview.csv').is_file():
    import spacy
    import linguistic_features as extraction
    from linguistic_fp_growth import sha256
    from annotation_preview import write_annotation_preview
    archive = ROOT / f"unsupervised-learning/data/Fake.br-Corpus-{manifest['corpus']['revision']}.zip"
    assert sha256(archive) == manifest['corpus']['archive_sha256']
    records = extraction.load_records(archive)
    nlp = spacy.load('pt_core_news_sm', disable=['ner'])
    preview_ids = pd.Series(train_ids).sample(n=12, random_state=SEED).tolist()
    write_annotation_preview(records, nlp, RUN, record_ids=preview_ids)
preview = pd.read_csv(RUN / 'annotation_preview.csv')
display(preview.groupby('record_id', sort=False).head(5))
display(Markdown(f'Prévia dos textos e anotações: `{RUN / "annotation_preview.md"}`'))

,record_id,start,end,text,pos,dep,head,lexicaleligible
0,fake/1692,0,4,Mais,ADV,advmod,1,True
1,fake/1692,5,8,uma,NUM,nummod,2,True
2,fake/1692,9,17,mensagem,NOUN,nsubj:pass,5,True
3,fake/1692,18,28,enigmática,ADJ,amod,2,True
4,fake/1692,29,32,foi,AUX,aux:pass,5,True
60,fake/1165,0,7,General,NOUN,ROOT,0,True
61,fake/1165,8,10,do,ADP,case,2,True
62,fake/1165,11,19,Exército,NOUN,nmod,0,True
63,fake/1165,19,20,:,PUNCT,punct,0,False
64,fake/1165,21,22,"""",PROPN,appos,0,False


Prévia dos textos e anotações: `C:\Users\CUL7CA\Desktop\Eldorado\olimpo-fake-news-ai\machine-learning\outputs\model-comparison\fp-growth-linguistic-20261006T230752Z\annotation_preview.md`

## 9. Classe e autoria: descrição posterior à descoberta

Só depois de congelar as regras, examinamos a composição Fake/True e a autoria. Os rótulos abaixo vêm da origem do corpus; não são avaliações das anotações linguísticas. A composição de um padrão é mostrada por partição e estrato de autoria, com o baseline do próprio estrato.

`lift_fake_descriptive` é uma comparação de composição de classe, **distinta do lift de coocorrência das regras**. Estratos com poucos exemplos da classe minoritária tornam essa leitura instável. Estas tabelas não refazem mineração ou discretização e não são usadas para escolher regras.

In [11]:
from linguistic_posthoc import write_posthoc

if not ((RUN / 'posthoc_pattern_composition.csv').is_file() and
        (RUN / 'posthoc_feature_distributions.csv').is_file()):
    write_posthoc(RUN)
composition = pd.read_csv(RUN / 'posthoc_pattern_composition.csv')
posthoc_rates = pd.read_csv(RUN / 'posthoc_feature_distributions.csv')
frame = metrics['corrected_pos_dep']
if not frame.empty:
    candidate = frame.sort_values(['validation_support', 'validation_confidence'],
                                  ascending=False).iloc[0]
    signature = tuple(sorted(set(json.loads(candidate['antecedents']) +
                                 json.loads(candidate['consequents']))))
    selected = composition[composition['variant'].eq('corrected_pos_dep') &
                           composition['features'].map(lambda x: tuple(json.loads(x)) == signature)]
    display(Markdown('**Composição do padrão com maior suporte de validação — seleção sem classe:**'))
    display(selected[['partition', 'author_state', 'population', 'occurrences',
                      'fake_count', 'true_count', 'fake_pct', 'baseline_fake_pct',
                      'lift_fake_descriptive', 'delta_fake_descriptive']])
display(posthoc_rates[posthoc_rates['partition'].eq('validation') &
                     posthoc_rates['author_state'].eq('all') &
                     posthoc_rates['feature'].str.startswith('POS_')])

**Composição do padrão com maior suporte de validação — seleção sem classe:**

,partition,author_state,population,occurrences,fake_count,true_count,fake_pct,baseline_fake_pct,lift_fake_descriptive,delta_fake_descriptive
495,train,all,4320,1231,486,745,0.394801,0.500000,0.789602,-0.105199
496,train,com_autor,2157,743,9,734,0.012113,0.020399,0.593815,-0.008286
497,train,sem_autor,2163,488,477,11,0.977459,0.978271,0.999170,-0.000812
498,validation,all,1440,416,149,267,0.358173,0.500000,0.716346,-0.141827
499,validation,com_autor,723,268,6,262,0.022388,0.023513,0.952151,-0.001125
500,validation,sem_autor,717,148,143,5,0.966216,0.980474,0.985458,-0.014258
501,test,all,1440,426,157,269,0.368545,0.500000,0.737089,-0.131455
502,test,com_autor,719,268,1,267,0.003731,0.015299,0.243894,-0.011568
503,test,sem_autor,721,158,156,2,0.987342,0.983356,1.004053,0.003985


,partition,origin,author_state,feature,records,present,missing,mean,median,q25,q75
126,validation,fake,all,POS_ADJ_rate,720,720,0,0.048385,0.041667,0.020408,0.065217
127,validation,fake,all,POS_ADV_rate,720,720,0,0.034237,0.022222,0.018868,0.051293
128,validation,fake,all,POS_NOUN_rate,720,720,0,0.212982,0.212766,0.180000,0.244898
129,validation,fake,all,POS_PROPN_rate,720,720,0,0.163316,0.160000,0.113636,0.208333
130,validation,fake,all,POS_VERB_rate,720,720,0,0.136846,0.133333,0.106953,0.163636
131,validation,fake,all,POS_PRON_rate,720,720,0,0.030234,0.021277,0.000000,0.041667
189,validation,true,all,POS_ADJ_rate,720,720,0,0.048238,0.041667,0.020408,0.066667
190,validation,true,all,POS_ADV_rate,720,720,0,0.022885,0.020000,0.000000,0.039216
191,validation,true,all,POS_NOUN_rate,720,720,0,0.241622,0.244671,0.200000,0.276596
192,validation,true,all,POS_PROPN_rate,720,720,0,0.163853,0.156863,0.104167,0.222222


## 10. Glossário e interpretação

| Item | Leitura |
|---|---|
| `POS_ADJ_rate` | Taxa de adjetivos. |
| `POS_ADV_rate` | Taxa de advérbios. |
| `POS_NOUN_rate` | Taxa de substantivos comuns. |
| `POS_PROPN_rate` | Taxa de nomes próprios. |
| `POS_VERB_rate` | Taxa de verbos, conforme a classificação do modelo. |
| `POS_PRON_rate` | Taxa de pronomes. |
| `DEP_nsubj_rate` | Taxa de sujeitos nominais. |
| `DEP_obj_rate` | Taxa de objetos diretos. |
| `DEP_amod_rate` | Taxa de modificadores adjetivais de nomes. |
| `DEP_advmod_rate` | Taxa de modificadores adverbiais. |
| `DEP_ccomp_rate` | Taxa de complementos oracionais com sujeito próprio. |
| `DEP_advcl_rate` | Taxa de orações adverbiais. |
| `DEP_complete_*` | Mesmas relações após o descarte conservador da última frase potencialmente cortada. |

Essas taxas descrevem a anotação gramatical da janela, sem medir diretamente subjetividade, falsidade ou qualidade factual. Sua contribuição é produzir perfis por notícia, com denominadores e procedência auditáveis. Mais regras não significam melhor detector. Priorize regras com ocorrências suficientes fora do treino e intervalos compatíveis com o efeito observado; confirme exemplos anotados antes de lhes dar interpretação linguística forte.

In [12]:
display(Markdown(f'**Artefatos disponíveis em `{RUN}`:**'))
display(pd.DataFrame({'arquivo': [str(path.relative_to(RUN))
                                  for path in sorted(RUN.rglob('*')) if path.is_file()]}))

**Artefatos disponíveis em `C:\Users\CUL7CA\Desktop\Eldorado\olimpo-fake-news-ai\machine-learning\outputs\model-comparison\fp-growth-linguistic-20261006T230752Z`:**

,arquivo
0,annotation_preview.csv
1,annotation_preview.md
2,comparison_summary.md
3,comparisons.csv
4,corrected_pos\all_rules.csv
...,...
78,training_bootstrap_manifest.json
79,training_bootstrap_summary.csv
80,training_rediscovery_stability.csv
81,variant_summary.csv
